# 02 - Holistic Data Preparer (Parts B to H)

End-to-end preprocessing and feature engineering of the **Customer Credit Risk** dataset
(1000 customers, target = `default_flag`).

| Part | Topic |
|---|---|
| B | Data acquisition (CSV, JSON, SQL, API) |
| C | Data understanding and cleaning (profiling, missing values) |
| D | Outlier handling |
| E | Feature engineering (mixed variables, dates, encoding, binning) |
| F | Feature scaling |
| G | Transformations, feature construction, ColumnTransformer |
| H | Final deliverable (dataset + report) |

> Run `python src/generate_dataset.py` once if `Data/Raw` is empty.

In [ ]:
import json
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except ImportError:          # running as a plain script
    display = print

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
BASE = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
RAW, EXT, PROC = BASE / "Data" / "Raw", BASE / "Data" / "External", BASE / "Data" / "Processed"
REP, VIZ = BASE / "Reports", BASE / "Visualization"
for p in (PROC, REP, VIZ):
    p.mkdir(parents=True, exist_ok=True)


def save_fig(name):
    plt.tight_layout()
    plt.savefig(VIZ / name, dpi=150, bbox_inches="tight")
    plt.show()

---
# Part B - Data Acquisition

## Task 3 - Import datasets from multiple sources

In [ ]:
# 3a. CSV  -> main dataset (financial + behavioural + target)
df_csv = pd.read_csv(RAW / "Customer.csv")
print("CSV shape :", df_csv.shape)
display(df_csv.head())

In [ ]:
# 3b. JSON -> customer metadata (demographics)
with open(RAW / "customer.json", encoding="utf-8") as f:
    payload = json.load(f)
df_json = pd.json_normalize(payload["records"])
print("JSON shape:", df_json.shape)
display(df_json.head())

In [ ]:
# 3c. SQL -> loan repayment history (run the .sql dump in an in-memory SQLite DB)
conn = sqlite3.connect(":memory:")
conn.executescript((RAW / "customer.sql").read_text(encoding="utf-8"))
df_sql = pd.read_sql("SELECT customer_id, repayment_history FROM loan_repayment", conn)
conn.close()
print("SQL shape :", df_sql.shape)
display(df_sql.head())

In [ ]:
# 3d. API -> external economic indicators (region level)
# Put the URL of your mock API here (e.g. mockapi.io / json-server). If it is None or fails,
# the saved response in Data/External is used, so the notebook always runs.
import requests

API_URL = None


def fetch_economic_indicators(url=None):
    if url:
        try:
            r = requests.get(url, timeout=10)
            r.raise_for_status()
            print("Fetched from API:", url)
            return r.json()
        except Exception as e:
            print("API call failed, using saved response ->", e)
    with open(EXT / "economic_indicators.json", encoding="utf-8") as f:
        print("Loaded saved API response from Data/External")
        return json.load(f)


df_api = pd.DataFrame(fetch_economic_indicators(API_URL)["data"])
display(df_api)

In [ ]:
# 3e. Merge all sources into one raw dataset
cols_order = ["customer_id", "age", "gender", "region", "education_level", "employment_type",
              "annual_income", "loan_amount", "loan_purpose", "credit_score", "repayment_history",
              "transaction_count", "spending_ratio", "join_date",
              "inflation_rate", "unemployment_rate", "gdp_growth", "default_flag"]

df_raw = (df_csv
          .merge(df_json, on="customer_id", how="left")
          .merge(df_sql, on="customer_id", how="left")
          .merge(df_api, on="region", how="left"))[cols_order]

df_raw.to_csv(PROC / "01_merged_raw_dataset.csv", index=False)
print("Merged raw dataset:", df_raw.shape)
display(df_raw.head())

---
# Part C - Data Understanding and Cleaning

## Task 4 - Explore the dataset (`.info()`, `.describe()`)

In [ ]:
df_raw.info()

In [ ]:
display(df_raw.describe().T.round(2))
display(df_raw.describe(include=["object", "string"]).T)

In [ ]:
# More understanding: duplicates, missing values, categories, target balance, validity checks
print("Duplicate rows       :", df_raw.duplicated().sum())
print("Duplicate customer_id:", df_raw["customer_id"].duplicated().sum())
print("\nMissing values per column:")
display(df_raw.isna().sum()[lambda s: s > 0].to_frame("missing").assign(pct=lambda d: (d.missing / len(df_raw) * 100).round(2)))

for c in ["gender", "region", "education_level", "employment_type", "loan_purpose"]:
    print(f"\n{c}:")
    print(df_raw[c].value_counts(dropna=False).to_string())

print("\nTarget distribution (default_flag):")
print(df_raw["default_flag"].value_counts(normalize=True).round(3).to_string())

print("\nValidity checks")
print("credit_score outside 300-850:", (~df_raw["credit_score"].dropna().between(300, 850)).sum())
print("age outside 18-80          :", (~df_raw["age"].dropna().between(18, 80)).sum())

## Task 5 - Pandas Profiling (data quality report)

`ydata-profiling` (the new name of *pandas-profiling*) creates a full HTML report.
If it is not installed, a simple pandas based fallback report is created so the notebook still runs.

In [ ]:
def data_quality_report(d):
    rows = []
    for c in d.columns:
        s = d[c]
        r = {"column": c, "dtype": str(s.dtype), "non_null": int(s.notna().sum()),
             "missing_count": int(s.isna().sum()), "missing_pct": round(s.isna().mean() * 100, 2),
             "unique_values": int(s.nunique())}
        if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            x = s.dropna()
            q1, q3 = x.quantile([0.25, 0.75])
            iqr = q3 - q1
            r.update(min=round(x.min(), 3), max=round(x.max(), 3), mean=round(x.mean(), 3),
                     median=round(x.median(), 3), std=round(x.std(), 3), skewness=round(x.skew(), 3),
                     iqr_outliers=int(((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum()),
                     zscore_outliers=int((np.abs(stats.zscore(x)) > 3).sum()))
        rows.append(r)
    return pd.DataFrame(rows)


def build_profile(d, title, path):
    try:
        from ydata_profiling import ProfileReport
        ProfileReport(d, title=title, explorative=True).to_file(path)
        return "ydata-profiling"
    except Exception:
        css = ("body{font-family:Arial,sans-serif;margin:30px}table{border-collapse:collapse}"
               "td,th{border:1px solid #ccc;padding:4px 8px;font-size:13px}th{background:#f0f0f0}")
        html = ("<html><head><meta charset='utf-8'><title>" + title + "</title><style>" + css + "</style></head><body>"
                "<h1>" + title + "</h1><p><i>Fallback pandas profile "
                "(install <code>ydata-profiling</code> for the full interactive report).</i></p>"
                "<h2>Overview</h2><ul><li>Rows: " + str(d.shape[0]) + "</li><li>Columns: " + str(d.shape[1]) +
                "</li><li>Missing cells: " + str(int(d.isna().sum().sum())) + "</li><li>Duplicate rows: " +
                str(int(d.duplicated().sum())) + "</li></ul><h2>Data quality per column</h2>" +
                data_quality_report(d).to_html(index=False) + "<h2>Descriptive statistics</h2>" +
                d.describe(include="all").T.round(3).to_html() + "</body></html>")
        Path(path).write_text(html, encoding="utf-8")
        return "pandas fallback"


dq_raw = data_quality_report(df_raw)
dq_raw.to_csv(REP / "data_quality_report.csv", index=False)
engine = build_profile(df_raw, "Raw Data Profiling Report", REP / "raw_profiling_report.html")
print("Profiling engine:", engine)
display(dq_raw)

In [ ]:
# Visual: missing values (before cleaning)
miss = df_raw.isna().mean().mul(100).sort_values()
miss = miss[miss > 0]
ax = miss.plot.barh(color="#d9534f", figsize=(7, 4))
ax.set_xlabel("% missing")
ax.set_title("Missing values per column (raw data)")
for i, v in enumerate(miss):
    ax.text(v + 0.1, i, f"{v:.1f}%", va="center")
save_fig("01_missing_values_before.png")

In [ ]:
# Basic cleaning: remove duplicate rows
n_before = len(df_raw)
df = df_raw.drop_duplicates().reset_index(drop=True)
assert df["customer_id"].is_unique
print(f"Removed {n_before - len(df)} duplicate rows -> shape {df.shape}")

## Task 6 - Handling missing data

Columns with missing values: `age`, `gender`, `employment_type`, `annual_income`, `loan_amount`, `credit_score`.

### 6a. Simple Imputer - numerical (mean / median)

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.preprocessing import RobustScaler

demo = df[["age"]].copy()
mean_imp = SimpleImputer(strategy="mean").fit_transform(demo).ravel()
median_imp = SimpleImputer(strategy="median").fit_transform(demo).ravel()
print("age mean   :", round(df["age"].mean(), 2), "| median:", df["age"].median())
print("age std  original:", round(df["age"].std(), 3), "| after mean imputation:", round(mean_imp.std(), 3),
      "| after median imputation:", round(median_imp.std(), 3))
print("-> mean/median imputation shrinks the variance a little. Age is fairly symmetric, so MEDIAN is used (robust).")

### 6b. Simple Imputer - categorical (most frequent) and 6c. Most frequent category imputation

In [ ]:
# 6b. SimpleImputer(strategy="most_frequent") on employment_type
emp_imp = SimpleImputer(strategy="most_frequent")
demo_emp = pd.Series(emp_imp.fit_transform(df[["employment_type"]]).ravel(), name="employment_type")
print("employment_type mode:", emp_imp.statistics_[0])
print(pd.concat([df["employment_type"].value_counts(dropna=False).rename("before"),
                 demo_emp.value_counts().rename("after")], axis=1))

# 6c. Most frequent category imputation (pandas way) on gender
mode_gender = df["gender"].mode()[0]
demo_gender = df["gender"].fillna(mode_gender)
print("\ngender mode:", mode_gender)
print(pd.concat([df["gender"].value_counts(dropna=False).rename("before"),
                 demo_gender.value_counts().rename("after")], axis=1))
print("-> gender is imbalanced, so mode-imputation pushes even more rows into 'Male' (bias risk).")

### 6d. Missing Indicator + Random Sample Imputation (`annual_income`)

In [ ]:
d = df.copy()
d["annual_income_missing"] = d["annual_income"].isna().astype(int)         # missing indicator
mask = d["annual_income"].isna()
random_sample = d["annual_income"].dropna().sample(mask.sum(), replace=True, random_state=RANDOM_STATE)
random_sample.index = d.index[mask]
d.loc[mask, "annual_income"] = random_sample

print("Missing before:", int(df["annual_income"].isna().sum()), "| after:", int(d["annual_income"].isna().sum()))
print("Mean/Std before:", round(df["annual_income"].mean()), round(df["annual_income"].std()),
      "| after:", round(d["annual_income"].mean()), round(d["annual_income"].std()))
print("Missing rate by employment type (why an indicator is useful):")
print(df.assign(m=df["annual_income"].isna()).groupby("employment_type")["m"].mean().round(3).to_string())
print("Default rate: income missing vs not ->",
      d.groupby("annual_income_missing")["default_flag"].mean().round(3).to_dict())

### 6e. KNN Imputer (multivariate) and 6f. MICE (IterativeImputer)

Both use the **other columns** to predict the missing values.
Features are scaled first (`RobustScaler`, robust to outliers) because KNN uses distances.

In [ ]:
num_cols = ["age", "annual_income", "loan_amount", "credit_score",
            "repayment_history", "transaction_count", "spending_ratio"]
target_cols = ["annual_income", "loan_amount", "credit_score"]
edu_rank = {"Primary": 0, "Secondary": 1, "Graduate": 2, "Post-Graduate": 3}


def scaled_impute(X, imputer):
    sc = RobustScaler()
    Z = imputer.fit_transform(sc.fit_transform(X))
    return pd.DataFrame(sc.inverse_transform(Z), columns=X.columns, index=X.index)


def make_feat(d):
    """Numeric matrix for the imputers: numeric columns + education / employment as helper features."""
    x = d[num_cols].copy()
    x["age"] = x["age"].fillna(x["age"].median())             # age is imputed first
    emp = d["employment_type"].fillna(d["employment_type"].mode()[0])
    x["education_rank"] = d["education_level"].map(edu_rank).astype(float)
    x["emp_self_employed"] = (emp == "Self-Employed").astype(float)
    x["emp_unemployed"] = (emp == "Unemployed").astype(float)
    return x


feat = make_feat(df)
knn_full = scaled_impute(feat, KNNImputer(n_neighbors=5))
mice_full = scaled_impute(feat, IterativeImputer(max_iter=10, random_state=RANDOM_STATE))

summary = pd.DataFrame({
    "original_mean": df[target_cols].mean(),
    "KNN_mean": knn_full[target_cols].mean(),
    "MICE_mean": mice_full[target_cols].mean(),
    "original_std": df[target_cols].std(),
    "KNN_std": knn_full[target_cols].std(),
    "MICE_std": mice_full[target_cols].std(),
}).round(1)
display(summary)

### 6g. How effective is each method? (masking experiment)

We hide 10 % of the **known** values, impute them with each method and compare with the truth.
Lower error = better.

In [ ]:
rs = np.random.RandomState(RANDOM_STATE)
masked, truth = feat.copy(), {}
for c in target_cols:
    obs = feat[c].dropna()
    lo, hi = obs.quantile([0.01, 0.99])
    eligible = obs[(obs >= lo) & (obs <= hi)].index          # keep extreme outliers out of the test
    idx = rs.choice(eligible, size=int(0.10 * len(obs)), replace=False)
    truth[c] = (idx, feat.loc[idx, c].copy())
    masked.loc[idx, c] = np.nan


def random_sample_imputer(X):
    out = X.copy()
    for c in X.columns:
        m = out[c].isna()
        out.loc[m, c] = out[c].dropna().sample(m.sum(), replace=True, random_state=RANDOM_STATE).values
    return out


def sk(strategy):
    return lambda X: pd.DataFrame(SimpleImputer(strategy=strategy).fit_transform(X), columns=X.columns, index=X.index)


methods = {
    "Mean": sk("mean"),
    "Median": sk("median"),
    "Random sample": random_sample_imputer,
    "KNN (k=5)": lambda X: scaled_impute(X, KNNImputer(n_neighbors=5)),
    "MICE": lambda X: scaled_impute(X, IterativeImputer(max_iter=10, random_state=RANDOM_STATE)),
}

rows = []
for name, fn in methods.items():
    imputed = fn(masked)
    for c in target_cols:
        idx, tv = truth[c]
        err = imputed.loc[idx, c] - tv
        rows.append({"method": name, "column": c, "MAE": err.abs().mean(), "RMSE": np.sqrt((err ** 2).mean())})

imp_cmp = pd.DataFrame(rows)
mae_table = imp_cmp.pivot(index="method", columns="column", values="MAE").round(1)
mae_table["avg_scaled_rank"] = mae_table.rank().mean(axis=1).round(2)
mae_table = mae_table.sort_values("avg_scaled_rank")
imp_cmp.to_csv(REP / "imputation_comparison.csv", index=False)
print("Mean Absolute Error on hidden values (lower is better):")
display(mae_table)

ax = imp_cmp.pivot(index="column", columns="method", values="MAE").pipe(
    lambda t: t.div(t.max(axis=1), axis=0)).plot.bar(figsize=(8, 4))
ax.set_ylabel("MAE relative to worst method")
ax.set_title("Imputation effectiveness (lower = better)")
save_fig("02_imputation_comparison.png")
best_per_col = mae_table[target_cols].idxmin()          # lowest error per column
print("Best method per column:", best_per_col.to_dict())

### 6h. Complete Case Analysis (dropping rows / columns)

In [ ]:
rows_all = len(df)
rows_cc = len(df.dropna())
print(f"Rows before: {rows_all} | after dropna(axis=0): {rows_cc} -> lost {rows_all - rows_cc} rows "
      f"({(rows_all - rows_cc) / rows_all:.1%})")
cols_cc = df.dropna(axis=1).shape[1]
print(f"Columns before: {df.shape[1]} | after dropna(axis=1): {cols_cc} -> lost {df.shape[1] - cols_cc} columns "
      f"({', '.join(df.columns[df.isna().any()])})")
print("Default rate - original:", round(df["default_flag"].mean(), 3),
      "| complete cases only:", round(df.dropna()["default_flag"].mean(), 3))
print("-> Complete case analysis throws away a large share of the data (and may bias it), so it is NOT used.")

### 6i. Final imputation strategy (applied to the dataset)

| Column | Method | Reason |
|---|---|---|
| `age` | SimpleImputer - median | few missing, roughly symmetric |
| `gender`, `employment_type` | SimpleImputer - most frequent | categorical |
| `annual_income` | **Missing indicator** column + the method with the lowest error in the experiment | missingness depends on employment type |
| `loan_amount`, `credit_score` | the method with the lowest error per column (experiment above) | data-driven choice |

In [ ]:
df_imp = df.copy()
df_imp["annual_income_missing"] = df_imp["annual_income"].isna().astype(int)

df_imp["age"] = SimpleImputer(strategy="median").fit_transform(df_imp[["age"]]).ravel()
for c in ["gender", "employment_type"]:
    df_imp[c] = SimpleImputer(strategy="most_frequent").fit_transform(df_imp[[c]]).ravel()

feat_final = make_feat(df_imp)
imputed_by_method = {m: methods[m](feat_final) for m in set(best_per_col)}
for c in target_cols:
    df_imp[c] = imputed_by_method[best_per_col[c]][c]

print("Method used per column:", best_per_col.to_dict())
print("Remaining missing values:", int(df_imp.isna().sum().sum()))
df_imp.to_csv(PROC / "02_after_imputation.csv", index=False)

---
# Part D - Outlier Handling

## Task 7 - Detect and treat outliers
Columns: `annual_income`, `loan_amount`, `credit_score` (plus validity rules for bounded columns).

In [ ]:
out_cols = ["annual_income", "loan_amount", "credit_score"]


def zscore_mask(s, thr=3):
    return ((s - s.mean()) / s.std()).abs() > thr


def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    i = q3 - q1
    return q1 - k * i, q3 + k * i


def iqr_mask(s, k=1.5):
    lo, hi = iqr_bounds(s, k)
    return (s < lo) | (s > hi)


def pct_mask(s, lo=0.01, hi=0.99):
    a, b = s.quantile([lo, hi])
    return (s < a) | (s > b)


det = pd.DataFrame({
    "Z-score (abs z > 3)": [int(zscore_mask(df_imp[c]).sum()) for c in out_cols],
    "IQR (1.5 x IQR)": [int(iqr_mask(df_imp[c]).sum()) for c in out_cols],
    "Percentile (<1% or >99%)": [int(pct_mask(df_imp[c]).sum()) for c in out_cols],
}, index=out_cols)
print("Number of outliers detected by each method:")
display(det)
det.to_csv(REP / "outlier_detection_counts.csv")

print("credit_score outside the valid range 300-850:", int((~df_imp["credit_score"].between(300, 850)).sum()))
print("Skewness:", df_imp[out_cols].skew().round(2).to_dict())

In [ ]:
# Treatment of annual_income with the four techniques
from scipy.stats import mstats

s = df_imp["annual_income"]
z_lo, z_hi = s.mean() - 3 * s.std(), s.mean() + 3 * s.std()
i_lo, i_hi = iqr_bounds(s)
p_lo, p_hi = s.quantile([0.01, 0.99])

treated = {
    "Original": s,
    "Z-score capping": s.clip(z_lo, z_hi),
    "IQR capping": s.clip(i_lo, i_hi),
    "Percentile clipping (1-99)": s.clip(p_lo, p_hi),
    "Winsorization (1%, 2%)": pd.Series(np.asarray(mstats.winsorize(s.to_numpy(), limits=(0.01, 0.02))), index=s.index),
}
cmp_out = pd.DataFrame({k: {"min": v.min(), "max": v.max(), "mean": v.mean(), "std": v.std(), "skew": v.skew()}
                        for k, v in treated.items()}).T.round(2)
display(cmp_out)

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=False)
for ax, (name, ser) in zip(axes, treated.items()):
    sns.boxplot(y=ser, ax=ax, color="#5bc0de")
    ax.set_title(name, fontsize=10)
    ax.set_ylabel("")
fig.suptitle("annual_income - outlier treatment techniques")
save_fig("03_outlier_treatment_income.png")

### Final outlier treatment

| Column | Treatment | Reason |
|---|---|---|
| `annual_income`, `loan_amount` | **Winsorization** (1 % lower, 2 % upper) | extreme values are real-looking but distort scaling; we keep all rows |
| `credit_score` | **clip to the valid range 300-850** (domain rule) | values like 999 or 120 are impossible |
| `repayment_history` | no change (0-12 is valid), handled by binning later | |

> The IQR rule still flags the natural right tail of a skewed distribution after winsorization;
> that remaining skew is handled by the Yeo-Johnson / Box-Cox transforms in Part G.

In [ ]:
df_out = df_imp.copy()
for c in ["annual_income", "loan_amount"]:
    df_out[c] = np.asarray(mstats.winsorize(df_out[c].to_numpy(), limits=(0.01, 0.02)))
df_out["credit_score"] = df_out["credit_score"].clip(300, 850).round(0)

after = pd.DataFrame({
    "max_before": df_imp[out_cols].max(), "max_after": df_out[out_cols].max(),
    "skew_before": df_imp[out_cols].skew(), "skew_after": df_out[out_cols].skew(),
    "iqr_outliers_before": [int(iqr_mask(df_imp[c]).sum()) for c in out_cols],
    "iqr_outliers_after": [int(iqr_mask(df_out[c]).sum()) for c in out_cols],
}).round(2)
display(after)
after.to_csv(REP / "outlier_treatment_summary.csv")

fig, axes = plt.subplots(2, 3, figsize=(13, 6))
for j, c in enumerate(out_cols):
    sns.boxplot(x=df_imp[c], ax=axes[0, j], color="#d9534f"); axes[0, j].set_title(f"{c} - before")
    sns.boxplot(x=df_out[c], ax=axes[1, j], color="#5cb85c"); axes[1, j].set_title(f"{c} - after")
save_fig("04_outliers_before_after.png")

df_out.to_csv(PROC / "03_cleaned_dataset.csv", index=False)
print("Cleaned dataset saved:", df_out.shape, "| missing values:", int(df_out.isna().sum().sum()))

---
# Part E - Feature Engineering

## Task 8 - Handle variable types
### 8a. Mixed variables (numeric + categorical)

In [ ]:
df_fe = df_out.copy()

num_features = df_fe.select_dtypes(include="number").columns.tolist()
cat_features = df_fe.select_dtypes(exclude="number").columns.tolist()
print("Numerical  :", num_features)
print("Categorical:", cat_features)

# customer_id is a MIXED variable (letters + digits, e.g. CUST0042): split it
split = df_fe["customer_id"].str.extract(r"^([A-Za-z]+)(\d+)$")
split.columns = ["id_prefix", "id_number"]
display(pd.concat([df_fe["customer_id"], split], axis=1).head(3))
print("-> it is only an identifier, so it is NOT used as a model feature (kept just for traceability).")

print("\ngender (categorical):", df_fe["gender"].unique().tolist())
print("age (numerical) range:", df_fe["age"].min(), "-", df_fe["age"].max())
print("spending_ratio (numerical, skewed): skewness =", round(df_fe["spending_ratio"].skew(), 2),
      "(handled in Part G)")

### 8b. Date and time variables - extract Year, Month, Day, Weekday

In [ ]:
df_fe["join_date"] = pd.to_datetime(df_fe["join_date"])
df_fe["join_year"] = df_fe["join_date"].dt.year
df_fe["join_month"] = df_fe["join_date"].dt.month
df_fe["join_day"] = df_fe["join_date"].dt.day
df_fe["join_weekday"] = df_fe["join_date"].dt.dayofweek          # Monday = 0
df_fe["join_weekday_name"] = df_fe["join_date"].dt.day_name()

# customer tenure in months (relative to the dataset snapshot date)
snapshot = pd.Timestamp("2025-12-31")
df_fe["tenure_months"] = ((snapshot.year - df_fe["join_date"].dt.year) * 12
                          + (snapshot.month - df_fe["join_date"].dt.month))

display(df_fe[["join_date", "join_year", "join_month", "join_day", "join_weekday",
               "join_weekday_name", "tenure_months"]].head())

## Task 9 - Encoding categorical variables
### 9a. Ordinal Encoding (`education_level`)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

edu_order = ["Primary", "Secondary", "Graduate", "Post-Graduate"]
ord_enc = OrdinalEncoder(categories=[edu_order])
df_fe["education_level_ord"] = ord_enc.fit_transform(df_fe[["education_level"]]).ravel().astype(int)
print(df_fe[["education_level", "education_level_ord"]].drop_duplicates().sort_values("education_level_ord").to_string(index=False))

### 9b. Label Encoding (`gender`) and 9c. One-Hot Encoding (`region`, `loan_purpose`)

In [ ]:
le = LabelEncoder()
df_fe["gender_label"] = le.fit_transform(df_fe["gender"])
print("Label mapping:", {k: int(v) for k, v in zip(le.classes_, le.transform(le.classes_))})
print("Note: gender has 3 classes, so label encoding implies an order. Fine for tree models; "
      "linear models would prefer one-hot.")

# One-hot encoding demo (the final pipeline in Part G does it with sklearn's OneHotEncoder)
ohe_demo = pd.get_dummies(df_fe[["region", "loan_purpose", "employment_type"]], prefix_sep="_", dtype=int)
print("\nOne-hot columns:", ohe_demo.columns.tolist())
display(pd.concat([df_fe[["region", "loan_purpose"]], ohe_demo.filter(regex="^(region|loan_purpose)")], axis=1).head())

## Task 10 - Encoding numerical features
### 10a. Binning (discretise income) and `repayment_history`

In [ ]:
income_bins = [0, 300_000, 600_000, 1_200_000, np.inf]
income_labels = ["Low (<3L)", "Medium (3-6L)", "High (6-12L)", "Very high (>12L)"]
df_fe["income_band"] = pd.cut(df_fe["annual_income"], bins=income_bins, labels=income_labels)
df_fe["income_band_code"] = df_fe["income_band"].cat.codes

rep_bins = [-1, 0, 2, 5, 12]
rep_labels = ["None (0)", "Low (1-2)", "Medium (3-5)", "High (6+)"]
df_fe["repayment_bucket"] = pd.cut(df_fe["repayment_history"], bins=rep_bins, labels=rep_labels)
df_fe["repayment_bucket_code"] = df_fe["repayment_bucket"].cat.codes

for col in ["income_band", "repayment_bucket"]:
    print(df_fe.groupby(col, observed=True)["default_flag"].agg(customers="count", default_rate="mean").round(3), "\n")

### 10b. Binarization (credit score > 700), 10c. Quantile binning and 10d. K-Means binning (`transaction_count`)

In [ ]:
from sklearn.preprocessing import Binarizer, KBinsDiscretizer

# Binarization: 1 if credit_score > 700
df_fe["credit_score_gt_700"] = Binarizer(threshold=700).fit_transform(df_fe[["credit_score"]]).ravel().astype(int)

# Quantile binning: each bin has (about) the same number of customers
kb_q = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile", random_state=RANDOM_STATE)
df_fe["txn_quantile_bin"] = kb_q.fit_transform(df_fe[["transaction_count"]]).ravel().astype(int)

# K-Means binning: bin edges are placed using 1-D k-means clusters
kb_k = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="kmeans", random_state=RANDOM_STATE)
df_fe["txn_kmeans_bin"] = kb_k.fit_transform(df_fe[["transaction_count"]]).ravel().astype(int)

print("Quantile bin edges:", kb_q.bin_edges_[0].round(1))
print("K-Means  bin edges:", kb_k.bin_edges_[0].round(1))
print("\ncredit_score > 700 -> default rate:")
print(df_fe.groupby("credit_score_gt_700")["default_flag"].agg(customers="count", default_rate="mean").round(3))
print("\nQuantile bins:")
print(df_fe.groupby("txn_quantile_bin")["default_flag"].agg(customers="count", default_rate="mean").round(3))
print("\nK-Means bins:")
print(df_fe.groupby("txn_kmeans_bin")["default_flag"].agg(customers="count", default_rate="mean").round(3))

---
# Part F - Feature Scaling

## Task 11 - Apply multiple scaling methods
Distance-based and gradient-based models (KNN, SVM, logistic regression, neural nets) need features on similar scales.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, Normalizer

scale_cols = ["age", "annual_income", "loan_amount", "credit_score",
              "repayment_history", "transaction_count", "spending_ratio"]

scalers = {
    "Standardization (z-score)": StandardScaler(),
    "Min-Max": MinMaxScaler(),
    "MaxAbs": MaxAbsScaler(),
    "Robust (median/IQR)": RobustScaler(),
    "Normalization (L2, row-wise)": Normalizer(norm="l2"),
}
scaled = {n: pd.DataFrame(s.fit_transform(df_fe[scale_cols]), columns=scale_cols) for n, s in scalers.items()}

summary = pd.concat({n: d.agg(["mean", "std", "min", "max"]).T for n, d in scaled.items()})
print("Standardization is used for annual_income and loan_amount (as per the task); all methods on all numeric columns:\n")
for c in ["annual_income", "loan_amount"]:
    print(c)
    display(summary.xs(c, level=1).round(3))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.ravel()
axes[0].hist(df_fe["annual_income"], bins=40, color="#777"); axes[0].set_title("Original")
for ax, (n, d) in zip(axes[1:], scaled.items()):
    ax.hist(d["annual_income"], bins=40, color="#5bc0de"); ax.set_title(n, fontsize=10)
fig.suptitle("annual_income under different scalers (shape stays the same, only the scale changes)")
save_fig("05_scaling_comparison.png")

pd.concat({n: d.agg(["mean", "std", "min", "max"]).T for n, d in scaled.items()}).round(4)\
    .to_csv(REP / "scaling_summary.csv")

| Scaler | Formula | Use when |
|---|---|---|
| Standardization | (x - mean) / std | roughly normal data; linear models, SVM, PCA |
| Min-Max | (x - min) / (max - min) | bounded 0-1 range; neural networks; no big outliers |
| MaxAbs | x / max(abs(x)) | sparse data, keeps zeros |
| Robust | (x - median) / IQR | data with outliers |
| Normalizer (L2) | each **row** divided by its norm | text / cosine similarity; rarely for tabular data |

---
# Part G - Feature Construction and Transformation

## Task 12 - Apply transformations
### 12a. `FunctionTransformer` - log, reciprocal and square root on `spending_ratio`

In [ ]:
from sklearn.preprocessing import FunctionTransformer, PowerTransformer

sr = df_fe[["spending_ratio"]].to_numpy()
func_versions = {
    "Original": sr,
    "Log  (log1p)": FunctionTransformer(np.log1p).fit_transform(sr),
    "Reciprocal (1/x)": FunctionTransformer(np.reciprocal).fit_transform(sr),
    "Square root": FunctionTransformer(np.sqrt).fit_transform(sr),
}
skew_sr = pd.Series({k: stats.skew(v.ravel()) for k, v in func_versions.items()}, name="skewness").round(3)
display(skew_sr.to_frame())

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (k, v) in zip(axes, func_versions.items()):
    ax.hist(v.ravel(), bins=35, color="#5bc0de")
    ax.set_title(f"{k}\nskew = {skew_sr[k]}", fontsize=10)
fig.suptitle("spending_ratio - FunctionTransformer")
save_fig("06_function_transformers_spending_ratio.png")
print("Best (closest to 0 skew):", skew_sr.abs().idxmin())

### 12b. `PowerTransformer` - Box-Cox and Yeo-Johnson on `loan_amount` and `annual_income`

In [ ]:
rows, versions = [], {}
for c in ["loan_amount", "annual_income"]:
    x = df_fe[[c]].to_numpy()
    bc = PowerTransformer(method="box-cox")             # needs strictly positive values
    yj = PowerTransformer(method="yeo-johnson")         # works with zero / negative values too
    versions[c] = {"Original": x.ravel(), "Box-Cox": bc.fit_transform(x).ravel(), "Yeo-Johnson": yj.fit_transform(x).ravel()}
    rows.append({"column": c, "skew_original": stats.skew(x.ravel()),
                 "skew_boxcox": stats.skew(versions[c]["Box-Cox"]), "lambda_boxcox": bc.lambdas_[0],
                 "skew_yeojohnson": stats.skew(versions[c]["Yeo-Johnson"]), "lambda_yeojohnson": yj.lambdas_[0]})
power_tbl = pd.DataFrame(rows).set_index("column").round(3)
display(power_tbl)
power_tbl.to_csv(REP / "power_transform_summary.csv")

fig, axes = plt.subplots(2, 3, figsize=(14, 6))
for i, c in enumerate(versions):
    for j, (k, v) in enumerate(versions[c].items()):
        axes[i, j].hist(v, bins=35, color="#5cb85c" if j else "#777")
        axes[i, j].set_title(f"{c} - {k}", fontsize=10)
save_fig("07_power_transformers.png")

## Task 13 - Construct new features

| New feature | Formula | Meaning |
|---|---|---|
| `debt_to_income` | `loan_amount / annual_income` | how large the loan is compared to yearly income |
| `avg_monthly_transactions` | `transaction_count / 6` | the transaction count covers 6 months |
| `spending_to_income` | `spending_ratio / 100` | spending-to-income ratio as a fraction (the dataset gives it in %) |
| `disposable_income` | `annual_income x (1 - spending_to_income)` | income left after spending |

In [ ]:
df_fe["debt_to_income"] = df_fe["loan_amount"] / df_fe["annual_income"]
df_fe["avg_monthly_transactions"] = df_fe["transaction_count"] / 6
df_fe["spending_to_income"] = df_fe["spending_ratio"] / 100
df_fe["disposable_income"] = df_fe["annual_income"] * (1 - df_fe["spending_to_income"])

display(df_fe[["loan_amount", "annual_income", "debt_to_income", "transaction_count",
               "avg_monthly_transactions", "spending_to_income", "disposable_income"]].describe().T.round(3))

# Are the new features useful?  ->  univariate ROC-AUC against default_flag (0.5 = useless, 1.0 = perfect)
from sklearn.metrics import roc_auc_score

check = ["loan_amount", "annual_income", "debt_to_income", "transaction_count", "avg_monthly_transactions",
         "spending_ratio", "disposable_income", "credit_score", "repayment_history", "tenure_months", "age"]
auc_tbl = pd.DataFrame({
    "roc_auc": {c: roc_auc_score(df_fe["default_flag"], df_fe[c]) for c in check},
}).assign(separation=lambda d: (d["roc_auc"] - 0.5).abs() + 0.5,
          new_feature=lambda d: d.index.isin(["debt_to_income", "avg_monthly_transactions", "disposable_income"])
          ).round(3).sort_values("separation", ascending=False)
display(auc_tbl)
auc_tbl.to_csv(REP / "feature_usefulness_auc.csv")

### 12c. Column Transformer - different preprocessing for different columns (one pipeline)

| Group | Columns | Transformer |
|---|---|---|
| Skewed money features | `annual_income`, `loan_amount`, `spending_to_income`, `debt_to_income`, `disposable_income` | `PowerTransformer` (Yeo-Johnson, standardised) |
| Roughly symmetric numeric | `age`, `credit_score`, `tenure_months`, `join_year`, economic indicators | `StandardScaler` |
| Bounded / count features | `repayment_history`, `avg_monthly_transactions`, `join_month`, `join_day`, `join_weekday` | `MinMaxScaler` |
| Ordinal | `education_level` | `OrdinalEncoder` -> `MinMaxScaler` |
| Nominal | `region`, `loan_purpose`, `employment_type` | `OneHotEncoder` |
| Already encoded / flags | `gender_label`, `annual_income_missing`, `credit_score_gt_700`, bins | passthrough |

`customer_id` (identifier) and `default_flag` (target) are kept outside the transformer.
`transaction_count` and `spending_ratio` are dropped because `avg_monthly_transactions` / `spending_to_income`
carry exactly the same information (avoids perfect multicollinearity).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.base import clone
import joblib

power_cols = ["annual_income", "loan_amount", "spending_to_income", "debt_to_income", "disposable_income"]
std_cols = ["age", "credit_score", "tenure_months", "join_year", "inflation_rate", "unemployment_rate", "gdp_growth"]
minmax_cols = ["repayment_history", "avg_monthly_transactions", "join_month", "join_day", "join_weekday"]
ord_cols = ["education_level"]
ohe_cols = ["region", "loan_purpose", "employment_type"]
pass_cols = ["gender_label", "annual_income_missing", "credit_score_gt_700", "income_band_code",
             "repayment_bucket_code", "txn_quantile_bin", "txn_kmeans_bin"]
feature_cols = power_cols + std_cols + minmax_cols + ord_cols + ohe_cols + pass_cols

preprocessor = ColumnTransformer(
    transformers=[
        ("power", PowerTransformer(method="yeo-johnson"), power_cols),
        ("std", StandardScaler(), std_cols),
        ("minmax", MinMaxScaler(), minmax_cols),
        ("ordinal", Pipeline([("enc", OrdinalEncoder(categories=[edu_order])), ("scale", MinMaxScaler())]), ord_cols),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ohe_cols),
        ("keep", "passthrough", pass_cols),
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

X, y = df_fe[feature_cols], df_fe["default_flag"]
print("Input features:", X.shape[1], "| rows:", len(X))
print("Missing values in X:", int(X.isna().sum().sum()))
preprocessor

In [ ]:
# (1) Leakage-safe version for modelling: split first, fit the pipeline on TRAIN only
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
pre_split = clone(preprocessor).fit(X_tr)
train_df = pre_split.transform(X_tr)
test_df = pre_split.transform(X_te)
for d_, idx, yy in [(train_df, X_tr.index, y_tr), (test_df, X_te.index, y_te)]:
    d_.insert(0, "customer_id", df_fe.loc[idx, "customer_id"].values)
    d_["default_flag"] = yy.values

# (2) Full processed dataset (deliverable of the project): pipeline fitted on all rows
pre_full = clone(preprocessor).fit(X)
final_df = pre_full.transform(X)
final_df.insert(0, "customer_id", df_fe["customer_id"].values)
final_df["default_flag"] = y.values

print("Full processed dataset :", final_df.shape)
print("Train / Test           :", train_df.shape, "/", test_df.shape)
print("Default rate train/test:", round(y_tr.mean(), 3), "/", round(y_te.mean(), 3), "(stratified split)")
print("Missing values         :", int(final_df.isna().sum().sum()))
print("Non-numeric columns    :", final_df.drop(columns="customer_id").select_dtypes(exclude="number").columns.tolist())
display(final_df.head())

In [ ]:
# Before / after the final pipeline: skewness of the money features
skew_tbl = pd.DataFrame({
    "skew_before": [stats.skew(X[c]) for c in power_cols],
    "skew_after": [stats.skew(final_df[c]) for c in power_cols],
}, index=power_cols).round(3)
display(skew_tbl)
skew_tbl.to_csv(REP / "final_skewness_before_after.csv")

fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for j, c in enumerate(power_cols):
    axes[0, j].hist(X[c], bins=35, color="#d9534f"); axes[0, j].set_title(f"{c}\nbefore", fontsize=9)
    axes[1, j].hist(final_df[c], bins=35, color="#5cb85c"); axes[1, j].set_title("after (Yeo-Johnson)", fontsize=9)
save_fig("08_final_pipeline_before_after.png")

---
# Part H - Final Deliverable

## Task 14 - Final cleaned and transformed dataset

In [ ]:
final_df.to_csv(PROC / "final_processed_dataset.csv", index=False)
train_df.to_csv(PROC / "final_train.csv", index=False)
test_df.to_csv(PROC / "final_test.csv", index=False)
joblib.dump(pre_full, PROC / "preprocessing_pipeline.joblib")

# reports on the processed data
dq_final = data_quality_report(final_df)
dq_final.to_csv(REP / "data_quality_report_after_cleaning.csv", index=False)
engine = build_profile(final_df, "Processed Data Profiling Report", REP / "profiling_report.html")
print("Profiling engine:", engine)
print("Saved ->", sorted(p.name for p in PROC.iterdir()))

In [ ]:
# Visual summary of the final dataset
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
y.value_counts().sort_index().plot.bar(ax=axes[0], color=["#5cb85c", "#d9534f"])
axes[0].set_title("Target: default_flag (class imbalance)")
axes[0].set_xticklabels(["0 - No default", "1 - Default"], rotation=0)
corr = final_df.drop(columns="customer_id").corr()["default_flag"].drop("default_flag")
top = corr.reindex(corr.abs().sort_values(ascending=False).index)[:15][::-1]
top.plot.barh(ax=axes[1], color=["#d9534f" if v > 0 else "#5bc0de" for v in top])
axes[1].set_title("Top 15 features by correlation with default_flag")
save_fig("09_final_dataset_overview.png")

## Task 15 - Report

The code below writes `Reports/final_report.md` using the **actual numbers** produced in this notebook.

In [ ]:
def md_table(d, index=True):
    d = d.reset_index().rename(columns={"index": "name"}) if index else d
    head = "| " + " | ".join(map(str, d.columns)) + " |"
    sep = "|" + "---|" * len(d.columns)
    body = ["| " + " | ".join(str(v) for v in r) + " |" for r in d.itertuples(index=False)]
    return "\n".join([head, sep, *body])


miss_before = df.isna().sum()[lambda s: s > 0].to_frame("missing_before").assign(
    pct=lambda d: (d.missing_before / len(df) * 100).round(2))
miss_before["strategy"] = pd.Series({
    "age": "SimpleImputer (median)", "gender": "Most frequent", "employment_type": "SimpleImputer (most frequent)",
    "annual_income": "Missing indicator + " + best_per_col["annual_income"], "loan_amount": best_per_col["loan_amount"], "credit_score": best_per_col["credit_score"]})
miss_before["missing_after"] = 0

n_dups = int(df_raw.duplicated().sum())
cc_lost = len(df) - len(df.dropna())
n_new_cols = final_df.shape[1] - 2

report = f"""# Holistic Data Preparer - Final Report

**Dataset:** Customer Credit Risk (3 sources: CSV, JSON, SQL + external economic API)  
**Records:** {len(df_raw)} raw rows -> {len(df)} after removing {n_dups} duplicate rows  
**Target:** `default_flag` (0 = No default, 1 = Default) - default rate **{y.mean():.1%}**

## 1. Missing value strategies and their effectiveness

{md_table(miss_before)}

* **Complete case analysis** would drop **{cc_lost} of {len(df)} rows ({cc_lost / len(df):.1%})**, so it was rejected.
* To measure effectiveness, 10 % of known values of `annual_income`, `loan_amount` and `credit_score` were hidden and re-imputed.
  Mean Absolute Error (lower is better):

{md_table(mae_table)}

* Lowest-error method per column (used in the final dataset): {", ".join(f"`{c}` -> **{m}**" for c, m in best_per_col.items())}.
  KNN and MICE use the relationship with other columns (loan, education, employment, ...); simple statistics are strong baselines for heavy-tailed money columns, so the choice was made from the experiment, not assumed.
* Mean / median imputation shrinks variance; random-sample imputation keeps the distribution but adds noise.
* A **missing-indicator** column (`annual_income_missing`) was added because income is missing more often for self-employed customers.

## 2. Outlier handling results

Outliers detected before treatment:

{md_table(det)}

After treatment:

{md_table(after)}

Note: the IQR rule still flags the natural right tail of skewed data after winsorization; the extreme values (maximums) and skewness dropped sharply, and the remaining skew is handled by Yeo-Johnson in the pipeline.

* `annual_income`, `loan_amount`: **Winsorization** (1 % lower / 2 % upper) - all rows kept, extreme values capped.
* `credit_score`: clipped to the valid range **300-850** (values such as 999 or 120 are impossible).
* Z-score is unreliable on skewed data (mean and std are themselves inflated by outliers); IQR and percentile are more robust.

## 3. Encoding methods

| Type | Columns | Method |
|---|---|---|
| Ordinal | `education_level` | OrdinalEncoder (Primary < Secondary < Graduate < Post-Graduate) |
| Label | `gender` | LabelEncoder |
| One-hot | `region`, `loan_purpose`, `employment_type` | OneHotEncoder |
| Binning | `annual_income`, `repayment_history` | manual bins (`income_band_code`, `repayment_bucket_code`) |
| Binarization | `credit_score` | `credit_score_gt_700` (threshold 700) |
| Quantile / K-Means binning | `transaction_count` | `txn_quantile_bin`, `txn_kmeans_bin` |
| Date | `join_date` | year, month, day, weekday, tenure in months |

## 4. Scaling and transformations - and why

* **Scalers compared:** Standardization, Min-Max, MaxAbs, Robust, Normalizer.
* **Skewed features** (`annual_income`, `loan_amount`, `spending_ratio`, ...) were transformed with **Yeo-Johnson** inside the pipeline:

{md_table(skew_tbl)}

* On `spending_ratio` the log / reciprocal / square-root skewness was:

{md_table(skew_sr.to_frame())}

  Closest to zero skew: **{skew_sr.drop("Original").abs().idxmin()}**. The reciprocal makes a right-skewed variable worse, so it is not used.

* Box-Cox and Yeo-Johnson on income and loan amount:

{md_table(power_tbl)}

* Symmetric columns -> StandardScaler; bounded / count columns -> MinMaxScaler.

## 5. Newly engineered features and their usefulness

{md_table(auc_tbl)}

* `debt_to_income` combines two weak features into one that separates defaulters better than either on its own.
* `avg_monthly_transactions` and `spending_to_income` are the same information as the originals in a more interpretable unit (no extra predictive power by themselves).
* `disposable_income` gives a view of affordability after spending.
* Bins and flags (`credit_score_gt_700`, `income_band_code`, ...) help linear and rule-based models.

## 6. Final dataset shape and readiness for ML

| Item | Value |
|---|---|
| Final dataset (`final_processed_dataset.csv`) | **{final_df.shape[0]} rows x {final_df.shape[1]} columns** ({n_new_cols} features + `customer_id` + `default_flag`) |
| Train / test (stratified 80/20) | {train_df.shape} / {test_df.shape} |
| Missing values | {int(final_df.isna().sum().sum())} |
| Non-numeric feature columns | 0 |
| Saved pipeline | `preprocessing_pipeline.joblib` |

**Readiness:** the data is complete, numeric, scaled and encoded. Remaining caution points:

* **Class imbalance** ({y.mean():.1%} defaulters) - use stratified splits, `class_weight="balanced"` or SMOTE, and judge models with recall / F1 / PR-AUC.
* **Data leakage** - imputation and winsorization were done on the full data for learning purposes; for production, fit every step on the training set only
  (the scaling / encoding pipeline in `final_train.csv` / `final_test.csv` already follows this rule).
* **Fairness** - `gender` and `region` are sensitive; check model bias before use.
* Next step: train baseline models (Logistic Regression, Random Forest, XGBoost) and evaluate.
"""
(REP / "final_report.md").write_text(report, encoding="utf-8")
print("Report written to Reports/final_report.md  (", len(report.split()), "words )")

## Project complete

Files created:

* `Data/Processed/` - merged raw, after imputation, cleaned, **final_processed_dataset.csv**, train/test, pipeline
* `Reports/` - data quality reports, profiling reports, imputation / outlier / scaling summaries, **final_report.md**
* `Visualization/` - all charts